In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    auc
)
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb


In [ ]:
dataset = Path('./output/feature_extraction/feature_dataset_labeled.csv')
df = pd.read_csv(dataset)

X = df.drop(columns=['file_name', 'cough_label', 'tb_label'])
y = df['cough_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

In [ ]:
model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train, y_train,
    sample_weight=sample_weight,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)


In [ ]:
# 3. Model Evaluation
THRESHOLD = 0.6
y_prob = model.predict_proba(X_test)
y_pred = (y_prob[:, 1] >= THRESHOLD).astype(int)

if len(model.classes_) > 2:
    auc_val = roc_auc_score(
        y_test, y_prob,
        multi_class='ovr',
        average='weighted'
    )
    auc_label = "AUC Score (Weighted OVR)"
else:
    auc_val = roc_auc_score(y_test, y_prob[:, 1])
    auc_label = "AUC Score"

print(f"{' EVALUATION REPORT ':=^40}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"{auc_label}: {auc_val:.4f}")
print("-" * 40)
print("Classification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=['Normal (0)', 'Cough (1)']
))
print("=" * 40)

plt.style.use("default")

# 🔴 ONLY CHANGE IS HERE (3 → 2)
fig, (ax2, ax3) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

# -----------------
# Confusion Matrix
# -----------------
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
    xticklabels=['Normal', 'Cough'],
    yticklabels=['Normal', 'Cough']
)

ax2.set_title("2. Confusion Matrix", fontsize=13, fontweight='bold')
ax2.set_xlabel("AI Predicted")
ax2.set_ylabel("Actual")

# -----------------
# ROC Curve
# -----------------
if len(model.classes_) <= 2:
    fpr, tpr, thresholds = roc_curve(y_test, y_prob[:, 1])
    roc_auc = auc(fpr, tpr)

    tn, fp, fn, tp = cm.ravel()
    current_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    current_tpr = tp / (tp + fn) if (tp + fn) > 0 else 0

    ax3.plot(
        fpr, tpr,
        color='darkorange', lw=2,
        label=f'ROC (AUC = {roc_auc:.4f})'
    )
    ax3.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')



    ax3.set_title("3. ROC Curve Analysis", fontsize=13, fontweight='bold')
    ax3.set_xlabel("False Positive Rate")
    ax3.set_ylabel("True Positive Rate")
    ax3.legend(loc="lower right")
    ax3.grid(alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
results_df = pd.DataFrame({
    'File_Name': df.loc[y_test.index, 'file_name'],
    'Actual': y_test.values,
    'Predicted': y_pred,
    'Probability_Cough': y_prob[:, 1]
})

# False Positives 
false_positives = results_df[(results_df['Actual'] == 0) & (results_df['Predicted'] == 1)]

# False Negatives 
false_negatives = results_df[(results_df['Actual'] == 1) & (results_df['Predicted'] == 0)]

print(f"{' ERROR ANALYSIS ':=^40}")
print(f"จำนวนที่ AI ขี้ระแวง (FP): {len(false_positives)} ไฟล์")
print(f"จำนวนที่ AI ปล่อยหลุด (FN): {len(false_negatives)} ไฟล์")
print("-" * 40)

print("\n[รายชื่อไฟล์ที่ - False Positives]")
print(false_positives[['File_Name', 'Probability_Cough']].sort_index(ascending=True).head(10))

print("\n[รายชื่อไฟล์ที่ - False Negatives]")
print(false_negatives[['File_Name', 'Probability_Cough']].sort_index(ascending=True).head(10))

In [ ]:
def optimize_mlp_threshold(y_true, y_prob, classes):
    """
    Optimizes the decision threshold and visualizes performance.
    """
    if len(classes) != 2:
        print("Note: Multi-class thresholding requires One-vs-Rest strategy.")
        return None

    fpr, tpr, thresholds = roc_curve(y_true, y_prob[:, 1])

    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    best_threshold = thresholds[best_idx]

    y_pred_default = (y_prob[:, 1] >= 0.5).astype(int)
    y_pred_optimized = (y_prob[:, 1] >= best_threshold).astype(int)

    acc_default = accuracy_score(y_true, y_pred_default)
    acc_optimized = accuracy_score(y_true, y_pred_optimized)
    
    improvement = ((acc_optimized - acc_default) / acc_default) * 100

    print(f"[*] Optimal MLP Threshold identified: {best_threshold:.4f}")
    print("\n--- MLP PERFORMANCE SUMMARY ---")
    print(f"Default Accuracy (0.50): {acc_default:.4f}")
    print(f"Optimized Accuracy ({best_threshold:.2f}): {acc_optimized:.4f}")
    print(f"Improvement: {improvement:+.2f}%")
    print("-" * 35)
    return best_threshold

# เรียกใช้งาน
best_mlp_thresh = optimize_mlp_threshold(y_test, y_prob, model.classes_)

In [ ]:
def sweep_thresholds(y_true, y_prob, start=0.1, end=0.9, step=0.1):
    """
    Iterates through thresholds and prints the impact on classification errors.
    """
    # Create the range of thresholds
    thresholds = np.arange(start, end + 0.1, step)
    
    print(f"{'Threshold':<12} | {'TN':<5} | {'FP (Type I)':<12} | {'FN (Type II)':<12} | {'TP':<5}")
    print("-" * 65)

    # Use only the probability of the positive class (column 1)
    pos_probs = y_prob[:, 1]

    for t in thresholds:
        # Apply threshold
        y_pred_t = (pos_probs >= t).astype(int)
        
        # Calculate Confusion Matrix
        # Note: ravel() returns tn, fp, fn, tp for binary classification
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_t).ravel()
        
        
        print(f"{t:.1f}          | {tn:<5} | {fp:<12} | {fn:<12} | {tp:<5}")


sweep_thresholds(y_test, y_prob)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance 

def calculate_mlp_feature_importance(model, X_val, y_val, feature_names):
    """
    Calculates feature importance for MLP using the Permutation method.
    """
    print("[*] Calculating Permutation Importance (this may take a moment)...")
    
    # 1. Compute Permutation Importance
    result = permutation_importance(
        model, X_val, y_val, 
        n_repeats=10, 
        random_state=42, 
        n_jobs=-1
    )
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance_mean': result.importances_mean,
        'importance_std': result.importances_std
    }).sort_values(by='importance_mean', ascending=False)

    # 3. Visualize
    plt.figure(figsize=(10, 8)) 
    top_15 = importance_df.head(15) 
    
    plt.barh(top_15['feature'], top_15['importance_mean'], 
             xerr=top_15['importance_std'], color='teal', align='center')
    
    plt.gca().invert_yaxis()
    plt.title("Top 15 Most Important Features (MLP)", fontsize=14)
    plt.xlabel("Decrease in Accuracy Score when shuffled")
    plt.tight_layout()
    plt.show()

    return importance_df


mlp_importances = calculate_mlp_feature_importance(
    model, X_test, y_test, X.columns.tolist()
)
